# Lesson 08 — Linear Algebra Intuition\n\n**What this notebook does:** turns a support ticket into a *vector* (a list of numbers), turns a batch of tickets into a *matrix* (a table of those lists), and uses the *dot product* to compute an urgency-style score for one ticket and then for many tickets at once. Every idea here is built with plain Python first, then with NumPy, so you can see that NumPy is not doing anything mysterious — it is doing the exact same arithmetic, just faster and with less typing.

## A ticket as a vector\n\nA **vector** is just an ordered list of numbers, where each position (called a **component**) means something specific. Take one support ticket and describe it with three numbers: how many words it has, how many exclamation marks it has, and how many words from an \"urgent word\" list it contains (words like *urgent*, *now*, *immediately*). That list of three numbers, in that order, is a vector.\n\nWe build it below as a plain Python list first, then as a NumPy array — the same numbers, two different containers."

In [ ]:
import numpy as np\n\nticket_text = \"My payment failed twice, I need this resolved right now!!\"\n\n# [word_count, exclamation_count, urgent_word_count]\nticket_as_list = [10, 2, 2]\nticket_as_vector = np.array(ticket_as_list)\n\nprint(ticket_as_list)\nprint(ticket_as_vector)\nprint(type(ticket_as_list), type(ticket_as_vector))

## Vector arithmetic: add and scale\n\nTwo things you can do to vectors: **add** two vectors together (add matching positions), and **scale** a vector by a number (multiply every position by that number). Both are done position-by-position — the first number only ever interacts with the first number, the second only with the second, and so on.\n\nSay a second ticket arrives with vector `[4, 0, 1]`. Adding the two vectors gives a combined total; scaling `ticket_as_vector` by `2` doubles every component, as if the same ticket's signals were twice as strong."

In [ ]:
second_ticket_vector = np.array([4, 0, 1])\n\ncombined = ticket_as_vector + second_ticket_vector\ndoubled = ticket_as_vector * 2\n\nprint(\"combined:\", combined)\nprint(\"doubled: \", doubled)

## The dot product: turning a vector into one number\n\nThe **dot product** of two same-length vectors is: multiply each matching pair of numbers, then add up all those products. It takes two lists of numbers and collapses them into a single number.\n\nGive each of our three ticket signals a **weight** — how much it should count toward an urgency score: word count matters a little (weight `0.1`), each exclamation mark matters more (weight `1.5`), and each urgent word matters most (weight `3`). The weight vector is `[0.1, 1.5, 3]`.\n\nFor `ticket_as_vector = [10, 2, 2]`, the dot product with the weights is:\n\n```\n(10 × 0.1) + (2 × 1.5) + (2 × 3)\n  =  1.0   +   3.0     +   6.0\n  =  10.0\n```\n\nThat single number, `10.0`, is an urgency score built entirely from a dot product. This is exactly the same shape of calculation as Lesson 1's word-counting classifier — a set of signals, each multiplied by how much it matters, then summed. We compute it three ways below: a hand-written loop, NumPy's `np.dot`, and the `@` operator, to show all three give the identical answer."

In [ ]:
weights = np.array([0.1, 1.5, 3])\n\n# hand-written loop version\nmanual_total = 0\nfor signal, weight in zip(ticket_as_vector, weights):\n    manual_total += signal * weight\n\n# NumPy versions\ndot_version = np.dot(ticket_as_vector, weights)\nat_version = ticket_as_vector @ weights\n\nprint(\"loop result:\", manual_total)\nprint(\"np.dot result:\", dot_version)\nprint(\"@ result:\", at_version)

## A matrix: many tickets, stacked into a table\n\nA **matrix** is a grid of numbers arranged in rows and columns. If a vector is one shopping list, a matrix is a whole spreadsheet of shopping lists — one row per list, one column per item. Here, each **row** is one ticket's vector `[word_count, exclamation_count, urgent_word_count]`, and each **column** is one signal, repeated down every ticket.\n\nFour tickets stacked this way form a 4-row, 3-column matrix — written as **shape `(4, 3)`**: 4 rows, 3 columns, in that order."

In [ ]:
tickets_matrix = np.array([\n    [10, 2, 2],   # \"My payment failed twice, I need this resolved right now!!\"\n    [4, 0, 1],    # \"urgent: order missing\"\n    [8, 0, 0],    # \"Do you have this item in a larger size\"\n    [15, 3, 3],   # \"This is unacceptable, I need a refund immediately!!! now!!\"\n])\n\nprint(tickets_matrix)\nprint(\"shape:\", tickets_matrix.shape)

## Matrix times vector: scoring every ticket at once\n\nHere is the payoff. Instead of computing one dot product per ticket by hand, multiply the whole **matrix** by the **weight vector** in one shot: `tickets_matrix @ weights`. NumPy takes each row of the matrix, dot-products it with `weights`, and hands back one score per row — four tickets in, four scores out, no loop written by us.\n\nFor this to work, the number of **columns** in the matrix must match the number of entries in the vector — 3 signals per ticket, 3 weights. That matching-length rule is the one rule to remember about matrix-vector multiplication; the next cell after this one shows what happens when it's broken."

In [ ]:
scores = tickets_matrix @ weights\n\nfor row, score in zip(tickets_matrix, scores):\n    print(f\"{row} -> score {score}\")\n\nprint(\"matrix shape:\", tickets_matrix.shape, \"weights shape:\", weights.shape, \"scores shape:\", scores.shape)

## Shapes must match — what happens when they don't\n\nIf the weight vector has the wrong number of entries — say only 2 weights for 3-column tickets — NumPy refuses and raises an error rather than silently computing something wrong. Seeing the real error message once makes the shape rule much easier to remember than being told about it."

In [ ]:
wrong_length_weights = np.array([0.1, 1.5])  # only 2 numbers, tickets_matrix has 3 columns\n\ntickets_matrix @ wrong_length_weights